# Segmentação semântica few-shot com VTM

**Pergunta experimental:** depois de meta-treinar em algumas classes semânticas, o Visual Token Matching consegue segmentar `chair` e `couch`, classes não vistas no meta-treino, usando somente 1, 5 ou 10 imagens anotadas?

Cada classe é uma tarefa binária de segmentação por pixel: a classe escolhida é foreground e todo o restante é background; pixels incertos são ignorados. O VTM recebe imagens e máscaras support e prevê a máscara de uma imagem query por correspondência de tokens visuais. O baseline usa o mesmo encoder congelado, mas ajusta diretamente um decoder nos supports.

Para caber em pouco espaço, o notebook usa apenas o Taskonomy `debug` (`allensville`) no disco temporário `/content`. Somente manifest, checkpoint e resultados pequenos são persistidos no Google Drive. A simplificação impede avaliar generalização entre prédios, mas mantém a pergunta principal de generalização entre classes. Ative uma GPU em **Runtime → Change runtime type** antes do treino.

## 1. Conectar o Google Drive

A célula abaixo monta seu Google Drive em `/content/drive`. O dataset ficará no disco temporário do Colab, mas o manifest, o checkpoint treinado e os resultados serão gravados no Drive para não desaparecerem quando a sessão terminar. O Colab solicitará autorização de acesso à sua conta.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Instalar o experimento

A célula seguinte clona o repositório do GitHub na primeira execução ou executa `git pull` se ele já estiver em `/content`. Em seguida, instala o pacote e suas dependências em modo editável. A última linha confirma as versões do PyTorch e do `timm` e mostra qual GPU está ativa. Se não aparecer uma GPU, altere o tipo de runtime antes do meta-treino.

In [ ]:
from pathlib import Path
import subprocess

repo = Path('/content/Visual-Token-Matching')
if not (repo / '.git').exists():
    subprocess.run([
        'git', 'clone',
        'https://github.com/NataLira1/Visual-Token-Matching.git',
        str(repo),
    ], check=True)
else:
    subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only'], check=True)

%cd /content/Visual-Token-Matching
!pip install -q -e ".[experiment]"
import torch, timm
print('torch', torch.__version__, 'timm', timm.__version__, 'gpu', torch.cuda.get_device_name(0))

## 3.1 Validar o download seletivo

Esta etapa baixa somente o Taskonomy `debug` para `/content/taskonomy`, valida a instalação do `omnidata-tools==0.0.23` e substitui sua integração obsoleta com Google Forms por um aceite explícito. Os links oficiais das licenças são exibidos e nenhum nome/e-mail é transmitido. **Os dados em `/content` são apagados quando o runtime termina.**

In [ ]:
!sudo apt-get -qq update && sudo apt-get -qq install -y aria2
import importlib

# O pacote também contém um módulo legado chamado
# omnidata_tools.starter_dataset que não funciona. O downloader usa este:
from omnidata_tools.dataset import starter_dataset
downloader = importlib.import_module('omnidata_tools.dataset.download')

required_licenses = {'omnidata', 'taskonomy'}
missing_licenses = required_licenses - starter_dataset.STARTER_DATA_LICENSES.keys()
if missing_licenses:
    raise RuntimeError(
        'Instalação incompatível do omnidata-tools; licenças ausentes: '
        + ', '.join(sorted(missing_licenses))
    )
print('Instalação do omnidata-tools validada.')

print('\nLeia as licenças antes de continuar:')
for component in ('omnidata', 'taskonomy'):
    print(f'- {component}: {starter_dataset.STARTER_DATA_LICENSES[component]}')
license_acceptance = input(
    "\nDigite ACEITO para confirmar que leu e aceita os termos acima: "
).strip()
if license_acceptance != 'ACEITO':
    raise RuntimeError('Termos não aceitos; download cancelado.')

# A implementação original tenta registrar o aceite em um Google Form antigo,
# que atualmente encerra o CLI com código 1. O aceite acima é explícito e a
# substituição abaixo evita transmitir dados pessoais para esse endpoint.
def local_license_confirmation(components, require_prompt, component_to_license, email, name):
    for component in sorted(set(components) | {'omnidata'}):
        if component not in component_to_license:
            raise RuntimeError(f'Licença ausente para {component}.')
    print('Licenças aceitas explicitamente nesta execução do notebook.')

downloader.licenses_clickthrough = local_license_confirmation
download_options = dict(
    domains=['rgb', 'segment_semantic'],
    components=['taskonomy'],
    subset='debug',
    split='all',
    dest='/content/taskonomy',
    dest_compressed='/content/taskonomy_compressed',
    connections_total=16,
    n_workers=4,
    agree_all=True,
)

print('\nExecutando dry-run...')
# keep_compressed=True evita outro bug da versão 0.0.23, que tenta apagar
# arquivos simulados pelo dry-run e produz FileNotFoundError.
downloader.download(**download_options, dryrun=True, keep_compressed=True)
dry_run_ok = True
print('Dry-run validado. A próxima célula pode ser executada.')

### 3.2 Realizar o download

Execute esta célula somente depois de o dry-run terminar com sucesso. Ela baixa e extrai RGB e máscaras semânticas de `allensville` em `/content/taskonomy`, verifica se arquivos foram produzidos e mostra alguns caminhos encontrados. Esses dados são temporários: não reinicie o runtime até concluir preparação, treino e avaliação.

In [ ]:
from pathlib import Path

if not globals().get('dry_run_ok', False):
    raise RuntimeError('Execute e valide primeiro a célula de dry-run.')

downloader.download(**download_options, dryrun=False)

data_root = Path('/content/taskonomy')
downloaded_files = [path for path in data_root.rglob('*') if path.is_file()]
if not downloaded_files:
    raise RuntimeError(
        'O downloader retornou sucesso, mas nenhum arquivo foi encontrado no destino.'
    )
print(f'Download validado: {len(downloaded_files)} arquivos encontrados.')
for path in downloaded_files[:10]:
    print(path)

## Preparação, meta-treino e avaliação

O modo compacto divide deterministicamente as imagens de `allensville` em 70% treino, 15% validação e 15% teste. Isso preserva a generalização entre tarefas/classes, mas não mede generalização entre prédios.

### 4. Criar a configuração da execução

Esta célula lê `taskonomy_vtm_compact.yaml`, cria a pasta persistente `vtm_semantic_experiment` no Drive e grava uma cópia da configuração em `/content`. A configuração determina classes, número de episódios, shots, seeds, caminhos e hiperparâmetros. O arquivo temporário será usado igualmente pelas três etapas seguintes. A pasta nova impede misturar checkpoints e métricas do protocolo anterior.

In [ ]:
from pathlib import Path
import yaml
config = yaml.safe_load(Path('configs/taskonomy_vtm_compact.yaml').read_text())
output_root = Path('/content/drive/MyDrive/vtm_semantic_experiment')
output_root.mkdir(parents=True, exist_ok=True)
runtime_config = Path('/content/vtm_taskonomy_compact_runtime.yaml')
runtime_config.write_text(yaml.safe_dump(config, sort_keys=False))
print('Configuração compacta:', runtime_config)
print('Dados temporários:', config['data']['root'])
print('Resultados persistentes:', output_root)
runtime_config

### 5. Preparar o manifest

A preparação procura pares correspondentes de imagem RGB e máscara, mede a cobertura de cada classe e cria um manifest reproduzível. Cada imagem é atribuída, por hash, a exatamente um split: 70% treino, 15% validação ou 15% teste. Confira na saída a quantidade de exemplos positivos por classe; classes com pouca cobertura poderão ser ignoradas posteriormente.

In [ ]:
# 1. Preparação: pareamento, cobertura e split 70/15/15 por hash.
!vtm-taskonomy prepare --config /content/vtm_taskonomy_compact_runtime.yaml

### 5.1 Conferir visualmente os rótulos semânticos

Antes de treinar, esta célula abre exemplos reais do manifest. Para cada tarefa exibida, a primeira coluna é a imagem RGB, a segunda é a máscara binária usada como ground truth (branco = classe; preto = restante) e a terceira sobrepõe a classe em vermelho. Essa verificação confirma que os IDs representam os objetos esperados e que RGB/máscara estão pareados. Se uma máscara não corresponder ao título, não prossiga com o treino.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image as PILImage
from vtm.data import load_manifest, select_records

records = load_manifest(config['data']['manifest'])
minimum = float(config['data']['min_positive_fraction'])
examples = []
# Prioriza as classes realmente inéditas, depois validação e meta-treino.
for group, split in (
    ('test_classes', 'test'),
    ('val_classes', 'val'),
    ('train_classes', 'train'),
):
    for class_name, class_id in config['data'][group].items():
        pool = select_records(records, split, int(class_id), minimum)
        if pool:
            record = max(pool, key=lambda item: item.coverage[str(class_id)])
            examples.append((class_name, int(class_id), record))
        if len(examples) == 6:
            break
    if len(examples) == 6:
        break

if not examples:
    raise RuntimeError('Nenhuma classe configurada foi encontrada nas máscaras.')

figure, axes = plt.subplots(len(examples), 3, figsize=(12, 3.2 * len(examples)))
axes = np.atleast_2d(axes)
for row, (class_name, class_id, record) in enumerate(examples):
    rgb = np.asarray(PILImage.open(record.rgb).convert('RGB'))
    raw_mask = np.asarray(PILImage.open(record.mask))
    if raw_mask.ndim == 3:
        raw_mask = raw_mask[..., 0]
    binary = raw_mask == class_id
    overlay = rgb.copy()
    overlay[binary] = (0.55 * overlay[binary] + 0.45 * np.array([255, 0, 0])).astype(np.uint8)
    coverage = 100 * binary.mean()
    axes[row, 0].imshow(rgb)
    axes[row, 0].set_title(f'RGB — {record.split}')
    axes[row, 1].imshow(binary, cmap='gray', vmin=0, vmax=1)
    axes[row, 1].set_title(f'{class_name} (ID bruto {class_id}) — {coverage:.1f}%')
    axes[row, 2].imshow(overlay)
    axes[row, 2].set_title('Sobreposição do ground truth')
    for axis in axes[row]:
        axis.axis('off')
plt.tight_layout()
plt.show()

### 6. Executar o meta-treino

Nesta etapa o modelo aprende episodicamente nas classes de meta-treino. Cada episódio executa os quatro componentes do VTM: encoder das imagens, encoder das máscaras, matching dos tokens e decoder da segmentação. O ViT visual permanece congelado; são treinados o label encoder, o matcher, o decoder e os biases das tarefas. A validação periódica escolhe o melhor checkpoint por IoU e o salva no Drive. Classes sem exemplos suficientes para formar um episódio são informadas e ignoradas automaticamente.

In [ ]:
# 2. Meta-treino: executa as quatro etapas do VTM em episódios.
!vtm-taskonomy train --config /content/vtm_taskonomy_compact_runtime.yaml

### 7. Adaptar e avaliar

A avaliação usa classes não empregadas no meta-treino. Para cada configuração de 1, 5 e 10 shots, o VTM congela todos os pesos e adapta somente quatro vetores de task bias. O baseline ajusta um decoder diretamente sobre os mesmos supports. Ambos são avaliados nas mesmas queries de teste. A etapa gera `results.csv`, `summary.csv`, `hypothesis.json`, painéis qualitativos e mapas de atenção. Arquivos corrompidos são ignorados e registrados em `corrupt_records.json`.

In [ ]:
# 3. Adaptação few-shot, baseline e avaliação.
!vtm-taskonomy evaluate --config /content/vtm_taskonomy_compact_runtime.yaml

### 8. Interpretar os resultados

A tabela `summary.csv` compara os métodos por número de shots. As colunas significam:

- **`method`**: `vtm` é o Visual Token Matching; `baseline` é o decoder ajustado diretamente nos supports.
- **`shots`**: número de imagens anotadas fornecidas como support para adaptar a classe nova. Menos shots significa uma tarefa mais difícil.
- **`iou_mean`**: Intersection over Union média do foreground, calculada como `TP / (TP + FP + FN)`. É a métrica principal; quanto maior, melhor.
- **`iou_std`**: variação da IoU entre classes e seeds avaliadas. Neste modo compacto, com somente a seed 0, ela representa principalmente a diferença entre as classes de teste.
- **`dice_mean`**: coeficiente Dice médio, `2TP / (2TP + FP + FN)`. Quanto maior, melhor; ele dá mais peso à sobreposição do foreground.
- **`dice_std`**: variação do Dice entre classes e seeds.
- **`precision_mean`**: entre os pixels previstos como objeto, proporção que realmente pertence ao objeto, `TP / (TP + FP)`. Precisão baixa indica excesso de falsos positivos.
- **`precision_std`**: variação da precisão entre classes e seeds.
- **`recall_mean`**: entre os pixels que realmente pertencem ao objeto, proporção encontrada pelo modelo, `TP / (TP + FN)`. Recall baixo indica que o modelo perdeu partes do objeto.
- **`recall_std`**: variação do recall entre classes e seeds.
- **`false_positive_rate_mean`**: proporção média de pixels incorretamente marcados como objeto em queries negativas. Aqui, quanto menor, melhor.
- **`false_positive_rate_std`**: variação da taxa de falso positivo entre classes e seeds.

O quadro da hipótese compara `iou_mean` do VTM e do baseline em 5-shot e 10-shot. **`supported: true`** significa que o VTM obteve IoU maior; **`false`** significa que a hipótese foi rejeitada naquele número de shots. `class_replacements` registra se alguma classe solicitada foi substituída por falta de cobertura. O JSON completo preserva também versões do ambiente, e o painel final permite comparar visualmente RGB, ground truth e previsões.

In [ ]:
import json
import pandas as pd
from IPython.display import display, Image, JSON
out = Path(config['experiment']['output_dir'])
print('Resumo das métricas:')
display(pd.read_csv(out / 'summary.csv'))

# hypothesis.json não é uma tabela homogênea: contém texto, dicionários,
# listas de substituições e informações do ambiente.
hypothesis = json.loads((out / 'hypothesis.json').read_text(encoding='utf-8'))
print('\nHipótese:', hypothesis['claim'])
hypothesis_table = (
    pd.DataFrame.from_dict(hypothesis.get('tests', {}), orient='index')
    .rename_axis('shots')
    .reset_index()
)
if not hypothesis_table.empty:
    display(hypothesis_table)
print('Substituições de classes:', hypothesis.get('class_replacements', []))
display(JSON(hypothesis))

panels = sorted((out / 'panels').glob('*.png'))
if panels:
    display(Image(filename=str(panels[0])))